# HuggingFace Transformers — State-of-the-Art NLP in 3 Lines

---

## What Is the HuggingFace Transformers Library?

HuggingFace `transformers` is the most popular ML library on earth (50M+ downloads/month). It provides:
- Access to **300,000+ pre-trained models** on the HuggingFace Hub (BERT, GPT, T5, LLaMA, Stable Diffusion, Whisper, and more)
- A **unified API** to use all these models: same `from_pretrained()` call for everything
- **`pipeline()`**: 3-line inference for common tasks (no ML knowledge required)
- **`Trainer`**: production-grade fine-tuning loop
- Support for **PyTorch, TensorFlow, and JAX** backends

### Real-World Analogy

Before HuggingFace, using a pre-trained AI model was like trying to drive a car that came in 50 different pieces from different manufacturers — you had to assemble the engine, seats, steering wheel separately, and none of them fit together well. HuggingFace is like a **car rental service**: pick any car (model), get in, and drive. The keys, fuel, and controls all work the same way regardless of the brand.

---

## The Transformer Architecture (Conceptual)

The Transformer ("Attention Is All You Need", 2017) revolutionized NLP. Key idea: **self-attention** — every token looks at every other token and learns what to pay attention to.

```
Input: "The cat sat on the mat"
Each word attends to all others:
"cat" learns it relates to "sat" (subject-verb)
"mat" learns it relates to "on" and "sat" (location of action)
```

Three architectures:
| Architecture | Models | Best for |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa, DistilBERT | Classification, NER, QA (understanding) |
| **Decoder-only** | GPT-2, LLaMA, Mistral | Text generation, chatbots |
| **Encoder-Decoder** | T5, BART, mT5 | Translation, summarization, QA with generation |

---

## Prerequisites

- Python basics
- Basic ML concepts (classification, fine-tuning)
- PyTorch basics (tensors, models) — recommended

---

## Table of Contents

1. Installation & Setup
2. `pipeline()` — Zero-Shot Inference
3. Tokenizers — How Text Becomes Numbers
4. Auto Classes — Loading Any Model
5. Fine-Tuning with `Trainer`
6. Text Generation
7. Question Answering
8. Summarization
9. Zero-Shot Classification
10. HuggingFace Hub — Find & Share Models
11. Mini Project — Sentiment Analysis Fine-Tuning
12. Common Pitfalls
13. Interview Q&A
14. Resources
15. Summary & What's Next

---

**Official Docs:** https://huggingface.co/docs/transformers/  
**HuggingFace Hub:** https://huggingface.co/models  
**Paper — Attention Is All You Need:** https://arxiv.org/abs/1706.03762  
**Paper — BERT:** https://arxiv.org/abs/1810.04805  
**YouTube — Transformers Explained (Andrej Karpathy):** https://www.youtube.com/watch?v=kCc8FmEb1nY  
**HuggingFace NLP Course (free):** https://huggingface.co/learn/nlp-course/  

## 1. Installation & Setup

```bash
pip install transformers datasets evaluate accelerate

# For PyTorch backend (recommended)
pip install torch

# For TensorFlow backend
pip install tensorflow
```

In [ ]:
transformerstry:
    from transformers import (pipeline, AutoTokenizer, AutoModel,
                              AutoModelForSequenceClassification,
                              AutoModelForTokenClassification,
                              AutoModelForQuestionAnswering,
                              AutoModelForSeq2SeqLM)
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    import warnings; warnings.filterwarnings("ignore")
    import transformers
    print(f"Transformers {transformers.__version__} | PyTorch {torch.__version__}")
except ImportError:
    raise SystemExit("Run: pip install transformers torch sentencepiece")


## 2. `pipeline()` — Zero-Shot Inference

The `pipeline()` function is the fastest way to use any pre-trained model. It wraps the full inference flow: tokenize → model → decode output.

Available tasks: `text-classification`, `token-classification` (NER), `question-answering`, `summarization`, `translation`, `text-generation`, `zero-shot-classification`, `image-classification`, `automatic-speech-recognition`, and more.

In [ ]:
# ---- Sentiment Analysis ----
print("=== Sentiment Analysis ===")
sentiment = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)

texts = [
    "This movie is absolutely fantastic! One of the best I've seen.",
    "Terrible waste of time. I want my 2 hours back.",
    "It was okay, nothing special but not bad either.",
    "I can't believe how good this is! Exceeded all expectations!"
]

results = sentiment(texts)
for text, result in zip(texts, results):
    print(f"  {result['label']:10s} ({result['score']:.3f}): {text[:60]}...")

In [ ]:
# ---- Named Entity Recognition ----
print("=== Named Entity Recognition ===")
ner = pipeline(
    'ner',
    model='dslim/bert-base-NER',
    aggregation_strategy='simple'  # merges consecutive tokens of same entity
)

ner_text = "Elon Musk announced that Tesla will build a new Gigafactory in Berlin, Germany, investing $5 billion."
entities = ner(ner_text)
for ent in entities:
    print(f"  [{ent['entity_group']:<4}] {ent['word']:<25} (score: {ent['score']:.3f})")

In [ ]:
# ---- Zero-Shot Classification (no fine-tuning needed!) ----
# The model classifies text into ANY categories you provide
print("=== Zero-Shot Classification ===")
zsc = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli'
)

articles = [
    ("The Federal Reserve raised interest rates by 25 basis points, citing persistent inflation.",
     ["economics", "sports", "technology", "entertainment", "politics"]),

    ("Scientists discovered a new exoplanet in the habitable zone of a nearby star system.",
     ["science", "sports", "politics", "economics", "entertainment"]),

    ("The quarterback threw three touchdowns in the fourth quarter to win the championship.",
     ["sports", "science", "economics", "technology", "entertainment"]),
]

for text, labels in articles:
    result = zsc(text, candidate_labels=labels)
    top = result['labels'][0]
    score = result['scores'][0]
    print(f"\nText: {text[:70]}...")
    print(f"→ Predicted: {top} ({score:.3f})")
    print(f"  All: {dict(zip(result['labels'], [f'{s:.2f}' for s in result['scores']]))}")

In [ ]:
# ---- Summarization ----
print("=== Summarization ===")
summarizer = pipeline(
    'summarization',
    model='facebook/bart-large-cnn',
    max_length=80,
    min_length=20
)

long_text = """
The Amazon rainforest, often referred to as the "lungs of the Earth," is the world's 
largest tropical rainforest, covering over 5.5 million square kilometers. It produces 
20% of the world's oxygen and is home to an estimated 10% of all species on Earth, 
including 40,000 plant species, 1,300 bird species, and 3,000 types of fish. 
The rainforest plays a crucial role in regulating the global climate by absorbing 
vast amounts of carbon dioxide. However, deforestation driven by agriculture, logging, 
and mining has destroyed approximately 17% of the original forest cover over the past 
50 years. Scientists warn that if deforestation continues at the current rate, the 
Amazon could reach a "tipping point" where it transforms from a carbon sink to a 
carbon source, dramatically accelerating climate change.
"""

summary = summarizer(long_text.strip())
print(f"Original length: {len(long_text.split())} words")
print(f"Summary length:  {len(summary[0]['summary_text'].split())} words")
print(f"Summary: {summary[0]['summary_text']}")

In [ ]:
# ---- Question Answering (Extractive) ----
print("=== Question Answering ===")
qa = pipeline(
    'question-answering',
    model='deepset/roberta-base-squad2'
)

context = """
PyTorch was developed by Meta AI Research and was first released in September 2016.
It provides automatic differentiation via a dynamic computation graph (define-by-run),
which makes debugging easier compared to TensorFlow's original static graph approach.
PyTorch is now the dominant framework for deep learning research, used in over 70% 
of machine learning papers published at top conferences.
"""

questions = [
    "Who developed PyTorch?",
    "When was PyTorch first released?",
    "What percentage of papers use PyTorch?",
]

for q in questions:
    result = qa(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {result['answer']} (confidence: {result['score']:.3f})\n")

## 3. Tokenizers — How Text Becomes Numbers

Before a transformer model can process text, it must be converted to numbers. **Tokenizers** do this in three steps:
1. **Tokenize**: split text into subwords (BPE, WordPiece, or SentencePiece)
2. **Encode**: map each subword to an integer ID
3. **Add special tokens**: `[CLS]` (start), `[SEP]` (separator/end), padding

Subword tokenization is the key insight: `"unbelievable"` might be split into `["un", "##believ", "##able"]`. This handles rare words and out-of-vocabulary terms without an explosion in vocabulary size.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

text = "The quick brown fox jumps over the unbelievably lazy dog."

# Basic tokenization
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")
print(f"  → Note 'unbelievably' split into: {[t for t in tokens if '##' in t or 'un' in t]}")

# Full encoding (returns input_ids, attention_mask, etc.)
encoding = tokenizer(text, return_tensors='pt')
print(f"\nInput IDs:      {encoding['input_ids'][0].tolist()}")
print(f"Attention Mask: {encoding['attention_mask'][0].tolist()}")
print(f"Token count:    {encoding['input_ids'].shape[1]} (includes [CLS] and [SEP])")

# Decode back to text
decoded = tokenizer.decode(encoding['input_ids'][0])
print(f"\nDecoded: {decoded}")

# Special tokens
print(f"\nSpecial tokens:")
print(f"  [CLS] token ID: {tokenizer.cls_token_id}")
print(f"  [SEP] token ID: {tokenizer.sep_token_id}")
print(f"  [PAD] token ID: {tokenizer.pad_token_id}")
print(f"  Vocabulary size: {tokenizer.vocab_size:,}")

In [ ]:
# Batching with padding and truncation
sentences = [
    "Short sentence.",
    "This is a much longer sentence that has more words in it and will need padding.",
    "Medium length sentence here."
]

# padding='max_length': pad all to same length
# truncation=True: truncate if longer than max_length
batch = tokenizer(
    sentences,
    padding=True,          # pad to longest in batch
    truncation=True,       # truncate if needed
    max_length=30,
    return_tensors='pt'
)

print("Batch encoding shapes:")
print(f"  input_ids:      {batch['input_ids'].shape}")
print(f"  attention_mask: {batch['attention_mask'].shape}")
print("\nAttention mask (1=real token, 0=padding):")
for i, mask in enumerate(batch['attention_mask']):
    print(f"  Sentence {i}: {mask.tolist()}")

## 4. Auto Classes — Loading Any Model

`Auto` classes automatically detect the model architecture and load the right class. They are the standard way to load models from the Hub.

In [ ]:
# Load a pre-trained BERT model and get embeddings
from transformers import AutoTokenizer, AutoModel

model_name = 'distilbert-base-uncased'  # smaller, faster version of BERT

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModel.from_pretrained(model_name)

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Get contextual embeddings
text = "The bank by the river bank charges high bank fees."
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

# last_hidden_state: (batch, seq_len, hidden_size)
last_hidden = outputs.last_hidden_state
print(f"\nHidden state shape: {last_hidden.shape}")
print(f"  → {last_hidden.shape[0]} batch × {last_hidden.shape[1]} tokens × {last_hidden.shape[2]} hidden dims")

# [CLS] embedding = sentence-level representation (used for classification)
cls_embedding = last_hidden[:, 0, :]  # first token
print(f"\n[CLS] embedding shape: {cls_embedding.shape}")

# Mean pooling = average all token embeddings (another sentence representation)
attention_mask = inputs['attention_mask']
mean_pooled = (last_hidden * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)
print(f"Mean-pooled embedding shape: {mean_pooled.shape}")

# Compute sentence similarity using CLS embeddings
sentences = [
    "The cat sat on the mat.",
    "A feline rested on the rug.",  # similar
    "The stock market crashed today."  # unrelated
]

def get_embedding(text):
    inp = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        out = model(**inp)
    return out.last_hidden_state[:, 0, :].numpy()  # CLS token

embs = [get_embedding(s) for s in sentences]

# Cosine similarity
def cosine_sim(a, b):
    return float(np.dot(a.flatten(), b.flatten()) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"\nSemantic similarity:")
print(f"  Sentence 1 vs 2 (similar): {cosine_sim(embs[0], embs[1]):.3f}")
print(f"  Sentence 1 vs 3 (different): {cosine_sim(embs[0], embs[2]):.3f}")

## 5. Fine-Tuning with `Trainer`

Fine-tuning = taking a pre-trained model and training it on your specific dataset with a much lower learning rate. The model keeps its general language understanding and learns your specific task.

The `Trainer` class handles the training loop, evaluation, checkpointing, and logging.

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from torch.utils.data import Dataset
import numpy as np

# ==================================================
# Synthetic sentiment dataset (positive/negative)
# ==================================================

positive_texts = [
    "Absolutely loved this! Best experience ever.",
    "Outstanding quality and fast delivery!",
    "Exceeded all expectations. Highly recommend.",
    "Perfect product, exactly as described.",
    "Amazing customer service and great value.",
    "Brilliant! Will definitely order again.",
    "Superb quality, worth every penny.",
    "Five stars, couldn't be happier!",
    "Fantastic product and excellent packaging.",
    "Really impressed with the quality!",
    "Great experience from start to finish.",
    "Love it! Works perfectly.",
]

negative_texts = [
    "Terrible product, broke after one day.",
    "Complete waste of money. Very disappointed.",
    "Awful quality. Would not recommend.",
    "Extremely poor customer service experience.",
    "Product looks nothing like the pictures.",
    "Broke immediately. Requesting a refund.",
    "Worst purchase I have ever made.",
    "Total junk. Avoid at all costs.",
    "Defective item and no help from support.",
    "Disgusting quality for the price.",
    "Very disappointed. Complete scam.",
    "Never buying from this seller again.",
]

all_texts  = positive_texts + negative_texts
all_labels = [1] * len(positive_texts) + [0] * len(negative_texts)

# Split
from sklearn.model_selection import train_test_split
tr_t, te_t, tr_l, te_l = train_test_split(all_texts, all_labels, test_size=0.3, random_state=42)

print(f"Train: {len(tr_t)} samples, Test: {len(te_t)} samples")

# ==================================================
# Custom Dataset class
# ==================================================
model_name = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(model_name)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(texts, truncation=True, padding=True,
                                    max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(tr_t, tr_l, tokenizer)
test_dataset  = ReviewDataset(te_t, te_l, tokenizer)

print(f"Sample item keys: {list(train_dataset[0].keys())}")

In [ ]:
import evaluate

# Load pre-trained model for sequence classification
ft_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # binary: positive (1) vs negative (0)
    id2label={0: 'NEGATIVE', 1: 'POSITIVE'},
    label2id={'NEGATIVE': 0, 'POSITIVE': 1}
)

# TrainingArguments — controls everything about training
training_args = TrainingArguments(
    output_dir='./review_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,          # much lower than training from scratch!
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=5,
    report_to='none'             # disable WandB logging
)

# Metrics
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Trainer
trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting fine-tuning...")
trainer.train()

# Evaluate
results = trainer.evaluate()
print(f"\nFine-tuned model accuracy: {results['eval_accuracy']:.4f}")

In [ ]:
# Test the fine-tuned model with pipeline
ft_pipeline = pipeline(
    'text-classification',
    model=ft_model,
    tokenizer=tokenizer,
    device=-1  # CPU
)

new_reviews = [
    "This is the worst product I have ever bought!",
    "Absolutely wonderful experience, will buy again!",
    "It is okay, does the job but nothing special."
]

print("Fine-tuned model predictions:")
for review, pred in zip(new_reviews, ft_pipeline(new_reviews)):
    print(f"  {pred['label']:10s} ({pred['score']:.3f}): {review}")

## 6. Text Generation with GPT-2

In [ ]:
# GPT-2 text generation
generator = pipeline(
    'text-generation',
    model='gpt2',
    max_new_tokens=80,
    pad_token_id=50256  # GPT-2's EOS token
)

prompts = [
    "The future of artificial intelligence is",
    "Once upon a time, a scientist discovered that"
]

for prompt in prompts:
    result = generator(
        prompt,
        do_sample=True,          # sampling (creative) vs greedy (deterministic)
        temperature=0.7,         # lower = more focused, higher = more random
        top_p=0.9,               # nucleus sampling: top 90% probability mass
        num_return_sequences=1
    )
    print(f"Prompt: {prompt}")
    print(f"Generated: {result[0]['generated_text']}")
    print()

## 10. HuggingFace Hub — Find & Share Models

The Hub has 300,000+ models organized by task, language, and architecture.

```python
# Find models for a specific task
from huggingface_hub import list_models
models = list(list_models(task='text-classification', library='transformers', limit=10))

# Load any model
model = AutoModel.from_pretrained('bert-base-multilingual-cased')   # multilingual
model = AutoModel.from_pretrained('facebook/wav2vec2-base')         # audio
model = AutoModel.from_pretrained('google/vit-base-patch16-224')    # vision

# Push your own model
model.push_to_hub('my-username/my-model-name')
tokenizer.push_to_hub('my-username/my-model-name')
```

**Popular model categories:**

| Task | Popular Models |
|---|---|
| Classification/NER | BERT, RoBERTa, DistilBERT, XLM-RoBERTa |
| Generation (chat) | LLaMA-3, Mistral, Falcon, Gemma |
| Summarization/QA | BART, T5, PEGASUS, FLAN-T5 |
| Embeddings | sentence-transformers, E5, GTE, bge |
| Image | ViT, CLIP, Stable Diffusion |
| Audio | Whisper, wav2vec2 |

## 11. Mini Project — Multi-Task NLP Analysis System

Build a complete NLP pipeline that processes news articles and performs: sentiment, entity extraction, summarization, and topic classification.

In [ ]:
# Multi-task NLP pipeline

# Initialize pipelines (in production, you'd load these once)
print("Loading NLP pipelines...")
sentiment_pipe = pipeline('sentiment-analysis',
                           model='distilbert-base-uncased-finetuned-sst-2-english')
ner_pipe       = pipeline('ner', model='dslim/bert-base-NER',
                            aggregation_strategy='simple')
zsc_pipe       = pipeline('zero-shot-classification',
                            model='facebook/bart-large-mnli')
print("All pipelines ready!")

articles = [
    {
        'title': 'OpenAI Raises $1B from Microsoft',
        'text': "OpenAI and Microsoft announced a massive $1 billion investment deal in Seattle. "
                "CEO Sam Altman stated the funds will accelerate GPT research. "
                "The partnership is groundbreaking for the AI industry."
    },
    {
        'title': 'Climate Protests Disrupt London',
        'text': "Environmental activists blocked key bridges in London causing traffic chaos on Friday. "
                "Police arrested 200 protesters near Westminster. "
                "The government condemned the disruption as harmful and counterproductive."
    }
]

topics = ['business', 'politics', 'science', 'sports', 'environment', 'technology']

print("\n" + "="*60)
for article in articles:
    text = article['text']
    print(f"\n📰 {article['title']}")
    print(f"   {text[:80]}...")

    # Sentiment
    sent = sentiment_pipe(text[:512])[0]
    print(f"\n   Sentiment: {sent['label']} ({sent['score']:.2f})")

    # NER
    entities = ner_pipe(text[:512])
    entity_summary = {}
    for e in entities:
        entity_summary.setdefault(e['entity_group'], []).append(e['word'])
    print(f"   Entities:")
    for etype, names in entity_summary.items():
        print(f"     {etype}: {names}")

    # Topic
    topic_result = zsc_pipe(text[:512], candidate_labels=topics)
    top_topics = list(zip(topic_result['labels'][:3], [f"{s:.2f}" for s in topic_result['scores'][:3]]))
    print(f"   Topics: {top_topics}")
    print("="*60)

## 12. Common Pitfalls

### Pitfall 1: Using the Wrong Model for the Task
- NER → use `bert-base-NER` not `bert-base-uncased` (which has no NER head)
- Text generation → use decoder-only (GPT) not encoder-only (BERT)
- Translation → use encoder-decoder (T5, mBART) not encoder-only

### Pitfall 2: Max Token Length Limits
BERT has a **512 token** limit. Longer texts must be chunked or truncated:
```python
encoding = tokenizer(text, truncation=True, max_length=512)
```

### Pitfall 3: Not Moving Model to GPU
```python
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}
```

### Pitfall 4: Forgetting `model.eval()` and `torch.no_grad()`
```python
model.eval()
with torch.no_grad():
    outputs = model(**inputs)  # 2-3x faster, much less memory
```

### Pitfall 5: Catastrophic Forgetting During Fine-Tuning
Use a very low learning rate (`2e-5` to `5e-5`) for fine-tuning transformers. High LR destroys pre-trained weights. Also freeze lower layers and only train the top few layers + task head.

## 13. Interview Q&A

---

**Q1: What is the self-attention mechanism in Transformers?**

> Self-attention allows each token in a sequence to attend to all other tokens. For each token, three vectors are computed: Query (Q), Key (K), and Value (V). The attention score between tokens i and j is `softmax(Q_i · K_j / sqrt(d_k))`. This score determines how much token i "borrows information" from token j. The output is a weighted sum of all Value vectors. **Why it matters**: (1) Captures long-range dependencies ("It" in "The cat...It" links back to "cat" regardless of distance), (2) Parallelizable (unlike RNNs which process sequentially), (3) Interpretable via attention weights.

---

**Q2: What is the difference between encoder-only, decoder-only, and encoder-decoder transformers?**

> - **Encoder-only (BERT, RoBERTa)**: Reads the full input bidirectionally (attends to both left and right context). Produces rich contextual representations. Best for: classification, NER, question answering (span extraction).
> - **Decoder-only (GPT-2, LLaMA, Mistral)**: Generates tokens auto-regressively using causal (left-only) attention. Each token only sees previous tokens. Best for: text generation, chatbots, few-shot learning.
> - **Encoder-Decoder (T5, BART)**: Encoder reads full input; decoder generates output attending to encoder states. Best for: translation, summarization, abstractive QA.

---

**Q3: What is fine-tuning and how does it differ from training from scratch?**

> A pre-trained model has learned general language understanding from billions of text tokens. **Fine-tuning** adapts this model to a specific downstream task by continuing training on task-specific labeled data with a very small learning rate (`2e-5` vs `1e-3` for training from scratch). The model preserves its general knowledge while learning task-specific patterns. Benefits: needs far fewer labeled examples (100s vs millions), trains in minutes vs weeks, achieves much higher accuracy than training from scratch on small datasets.

---

**Q4: What is BPE (Byte Pair Encoding) tokenization?**

> BPE is a subword tokenization algorithm used by GPT, RoBERTa, and others. It starts with individual characters and iteratively merges the most frequent adjacent pairs: `"running"` → `["run", "ning"]`. Advantages: handles any word including rare words and out-of-vocabulary terms (worst case: single characters), fixed vocabulary size (typically 32K-50K), balances between character-level (too many tokens) and word-level (too many OOV words). BERT uses WordPiece (similar concept), T5 uses SentencePiece.

---

**Q5: What does the `[CLS]` token represent in BERT?**

> BERT prepends a special `[CLS]` (classification) token to every input. After passing through all transformer layers, the `[CLS]` token's hidden state aggregates information from the entire sequence through self-attention, making it a useful sentence-level representation. For classification tasks, BERT's classification head takes this `[CLS]` embedding and passes it through a linear layer to produce class probabilities. Note: the quality of `[CLS]` as a sentence embedding is not as good as purpose-built sentence encoders (Sentence-Transformers uses mean pooling instead).

## 14. Resources

### Official
- **HuggingFace Docs:** https://huggingface.co/docs/transformers/
- **HuggingFace Hub:** https://huggingface.co/models
- **HuggingFace NLP Course (free):** https://huggingface.co/learn/nlp-course/

### Papers
- **Attention Is All You Need (Transformer):** https://arxiv.org/abs/1706.03762
- **BERT:** https://arxiv.org/abs/1810.04805
- **GPT-2:** https://openai.com/research/language-unsupervised
- **T5:** https://arxiv.org/abs/1910.10683
- **RoBERTa:** https://arxiv.org/abs/1907.11692

### Videos
- **Andrej Karpathy — Let's build GPT from scratch:** https://www.youtube.com/watch?v=kCc8FmEb1nY
- **3Blue1Brown — Attention in Transformers:** https://www.youtube.com/watch?v=eMlx5fFNoYc
- **HuggingFace YouTube channel:** https://www.youtube.com/@HuggingFace

## 15. Summary & What's Next

### What You Learned

| Concept | Key Takeaway |
|---|---|
| **`pipeline()`** | 3-line inference for any NLP task — just specify task + model |
| **Tokenizer** | Text → subword tokens → integer IDs + attention_mask |
| **Auto classes** | `AutoTokenizer`, `AutoModel`, `AutoModelForX` — load any model the same way |
| **BERT** | Encoder-only; bidirectional; best for understanding tasks |
| **GPT** | Decoder-only; causal; best for generation |
| **T5/BART** | Encoder-decoder; best for seq2seq (translation, summarization) |
| **Fine-tuning** | Low LR (`2e-5`), `Trainer` class, `compute_metrics`, `TrainingArguments` |
| **Embeddings** | `last_hidden_state[:, 0, :]` (CLS) or mean pooling for sentence representations |
| **Zero-shot** | BART-MNLI classifies into ANY categories without fine-tuning |

### What's Next

**Sentence Transformers** — dedicated library for high-quality sentence embeddings:
- Optimized for semantic similarity, search, and clustering
- `SentenceTransformer.encode()` for fast batch encoding
- Used in RAG (Retrieval-Augmented Generation) systems and semantic search